In [49]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# ── Plotly theme for PPT ────────────────────────────────────────────────────
PPT_COLORS = {
    "train": "#2196F3",  # blue
    "val": "#FF9800",  # orange
    "test": "#4CAF50",  # green
    "pred_train": "#90CAF9",  # light blue
    "pred_val": "#FFCC80",  # light orange
    "pred_test": "#A5D6A7",  # light green
    "split": "#9E9E9E",  # grey
    "trend": "#E91E63",  # pink
    "season": "#9C27B0",  # purple
    "residual": "#FF5722",  # deep orange
    "bg": "#FFFFFF",
    "grid": "#E0E0E0",
    "text": "#212121",
    "checkpoint": "#D32F2F",  # red — data quality events
}

PPT_LAYOUT = dict(
    font=dict(family="Arial", size=14, color=PPT_COLORS["text"]),
    paper_bgcolor=PPT_COLORS["bg"],
    plot_bgcolor=PPT_COLORS["bg"],
    width=1000,
    height=500,
    margin=dict(l=60, r=40, t=70, b=60),
    legend=dict(
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor=PPT_COLORS["grid"],
        borderwidth=1,
        font=dict(size=12),
    ),
    xaxis=dict(
        showgrid=True, gridcolor=PPT_COLORS["grid"], linecolor=PPT_COLORS["grid"]
    ),
    yaxis=dict(
        showgrid=True, gridcolor=PPT_COLORS["grid"], linecolor=PPT_COLORS["grid"]
    ),
)


def apply_ppt_layout(fig, title="", height=None):
    layout = PPT_LAYOUT.copy()
    if height:
        layout["height"] = height
    fig.update_layout(
        **layout, title=dict(text=title, font=dict(size=18, color=PPT_COLORS["text"]))
    )
    return fig


def save_fig(fig, name):
    """Save figure as HTML (interactive) and PNG (for PPT)."""
    fig.write_html(f"images/{name}.html")
    fig.write_image(f"images/{name}.png", scale=2)
    print(f"Saved: {name}.html / {name}.png")


# ── Data-quality checkpoints (source: ANP footnotes) ───────────────────────
# Each entry: (date_or_start, date_or_None_if_point, short_label, long_tooltip)
CHECKPOINTS = [
    # ── Política de preços Petrobras / Brasil ──────────────────────────────
    (
        pd.Timestamp("2016-10-03"),
        None,
        "(*1) Adoção PPI",
        "Out/2016 — Petrobras adota PPI (Preço de Paridade de Importação): preços passam a seguir Brent + câmbio",
    ),
    # ── Choques nacionais ──────────────────────────────────────────────────
    (
        pd.Timestamp("2018-05-27"),
        pd.Timestamp("2018-06-26"),
        "(*2) Greve caminhoneiros",
        "27/05–26/06/2018 — Greve dos caminhoneiros: vendas de diesel reduziram ~30%",
    ),
    (
        pd.Timestamp("2020-03-09"),
        pd.Timestamp("2020-04-30"),
        "(*3) COVID-19 + guerra de preços",
        "Mar/2020 — Guerra de preços Rússia-Arábia Saudita + pandemia COVID-19: Brent cai >65%; WTI atingiu -US$37",
    ),
    (
        pd.Timestamp("2022-02-24"),
        None,
        "(*4) Invasão Ucrânia",
        "24/02/2022 — Invasão russa da Ucrânia: Brent dispara para US$133 em 2 semanas (+30%)",
    ),
    (
        pd.Timestamp("2022-10-05"),
        None,
        "(*5) Corte OPEP+",
        "Out/2022 — OPEP+ anuncia corte de ~2 mb/dia: reversão de queda e retomada de alta nos preços",
    ),
    (
        pd.Timestamp("2023-05-16"),
        None,
        "(*6) Fim do PPI",
        "16/05/2023 — Petrobras abandona PPI (gov. Lula): queda imediata de ~R$0,44/L nas distribuidoras",
    ),
    (
        pd.Timestamp("2023-08-15"),
        None,
        "(*7) Reajuste de Preços",
        "Ago/2023 — Petrobras anuncia aumento de 25,8% no diesel e 16,2% na gasolina",
    ),
    (
        pd.Timestamp("2026-02-28"),
        None,
        "(*8) Assassinato de Khamenei",
        "Fez/2026 — Ataque dos EUA & Israel mata lider do Irã",
    ),
]


def _y_to_paper(y_val, y_min, y_max):
    """Convert a data-space y value to paper [0,1] coordinates."""
    if y_max == y_min:
        return 0.5
    return (y_val - y_min) / (y_max - y_min)


def add_checkpoints(fig, series, checkpoints=CHECKPOINTS):
    """Overlay checkpoint markers and shaded regions on an existing Plotly figure.

    Annotation y follows the series value at each checkpoint date (+ small offset)
    so labels sit close to the line instead of all stacking at the top.
    Works on plain go.Figure() — no subplot row/col needed.
    """
    if series is None or len(series) == 0:
        return fig
    t_min, t_max = series.index.min(), series.index.max()
    y_min, y_max = float(series.min()), float(series.max())
    y_pad = (y_max - y_min) * 0.08  # 8% of range above the point

    for start, end, label, tooltip in checkpoints:
        if start > t_max:
            continue

        if end is not None:
            # ── Range event: shaded band ──────────────────────────────────
            x0 = max(start, t_min)
            x1 = min(end, t_max)
            if x0 > x1:
                continue
            fig.add_shape(
                type="rect",
                xref="x",
                yref="paper",
                x0=x0,
                x1=x1,
                y0=0,
                y1=1,
                fillcolor="rgba(211,47,47,0.10)",
                line=dict(width=0),
            )
            # Anchor label y to the mean series value inside the band
            mask = (series.index >= x0) & (series.index <= x1)
            y_anchor = float(series[mask].mean()) if mask.any() else y_max
            y_paper = min(_y_to_paper(y_anchor + y_pad, y_min, y_max), 0.97)
            fig.add_annotation(
                xref="x",
                yref="paper",
                x=x0,
                y=y_paper,
                text=label,
                showarrow=True,
                arrowhead=0,
                arrowcolor=PPT_COLORS["checkpoint"],
                arrowwidth=1,
                ax=18,
                ay=0,
                font=dict(size=10, color=PPT_COLORS["checkpoint"]),
                xanchor="left",
                bgcolor="rgba(255,255,255,0.75)",
                hovertext=tooltip,
            )
        else:
            # ── Point event: dashed vertical line + star on curve ─────────
            if start < t_min:
                continue
            fig.add_shape(
                type="line",
                xref="x",
                yref="paper",
                x0=start,
                x1=start,
                y0=0,
                y1=1,
                line=dict(color=PPT_COLORS["checkpoint"], width=1.5, dash="dot"),
            )
            nearest_idx = series.index.get_indexer([start], method="nearest")[0]
            y_val = float(series.iloc[nearest_idx])
            fig.add_trace(
                go.Scatter(
                    x=[series.index[nearest_idx]],
                    y=[y_val],
                    mode="markers",
                    marker=dict(
                        symbol="star",
                        size=12,
                        color=PPT_COLORS["checkpoint"],
                        line=dict(width=1, color="white"),
                    ),
                    name=label,
                    hovertext=tooltip,
                    hoverinfo="text",
                    showlegend=True,
                )
            )
            y_paper = min(_y_to_paper(y_val + y_pad, y_min, y_max), 0.97)
            fig.add_annotation(
                xref="x",
                yref="paper",
                x=start,
                y=y_paper,
                text=label,
                showarrow=True,
                arrowhead=2,
                arrowcolor=PPT_COLORS["checkpoint"],
                arrowwidth=1,
                arrowsize=0.8,
                ax=18,
                ay=-20,
                font=dict(size=10, color=PPT_COLORS["checkpoint"]),
                xanchor="left",
                bgcolor="rgba(255,255,255,0.75)",
                hovertext=tooltip,
            )

    return fig

In [50]:
url = 'https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/precos-revenda-e-de-distribuicao-combustiveis/shlp/semanal/semanal-brasil-desde-2013.xlsx'
filename = Path(urlparse(url).path).name

if Path(filename).exists():
  print(f"File already exists")
else:
  !wget {url}

File already exists


### Functions

In [51]:
def calc_best_lags(serie, nlags=20, alpha=0.05):

    acf_vals, confint = acf(x=serie, nlags=nlags, alpha=alpha, fft=False)

    best_lags = np.where((acf_vals < confint[:, 0]) | (acf_vals > confint[:, 1]))[0]

    best_lags = best_lags[best_lags != 0]
    return best_lags


def is_stationary(series, alpha=0.05):
    series = series.dropna()

    result = adfuller(series)
    p_value = result[1]

    return p_value <= alpha


def find_differencing_order(series, alpha=0.05, max_d=5):
    current_series = series.copy()

    for d in range(max_d + 1):
        p_value = adfuller(current_series.dropna())[1]

        print(f"d={d}, p-value={p_value:.5f}")

        if p_value <= alpha:
            return d

        current_series = current_series.diff()

    raise ValueError(f"Series is still non-stationary after {max_d} differences.")


def forecast(train_series, test_series, order=(1, 1, 1)):
    history = train_series.copy()

    dates = []
    preds = []

    for timestamp, true_value in test_series.items():
        model = ARIMA(history, order=order, freq="7D")
        fitted_model = model.fit()

        prediction = fitted_model.forecast(steps=1).iloc[0]

        dates.append(timestamp)
        preds.append(prediction)

        history = pd.concat(
            [history, pd.Series([true_value], index=[timestamp])]
        ).sort_index()

        history = history.asfreq("7D").interpolate()

    forecast_series = pd.Series(preds, index=pd.to_datetime(dates))
    forecast_series.name = "forecast"

    return forecast_series


def significant_lags_acf(series, max_lag):
    acf_x, confint = acf(series, nlags=max_lag, alpha=0.05)

    # confint is centered on acf_x, re-center to zero
    lower = confint[:, 0] - acf_x
    upper = confint[:, 1] - acf_x

    selected = []

    for i in range(1, max_lag + 1):
        if acf_x[i] < lower[i] or acf_x[i] > upper[i]:
            selected.append(i)

    return selected


def window_data(window_size, train_series, test_series):
    full_series = np.concatenate((train_series, test_series))

    windows = np.lib.stride_tricks.sliding_window_view(full_series, window_size + 1)
    X = windows[:, :-1]
    y = windows[:, -1]

    X_train = X[: -len(test_series)]
    y_train = y[: -len(test_series)]
    X_test = X[-len(test_series) :]
    y_test = y[-len(test_series) :]

    return (X_train, y_train), (X_test, y_test)


# ── Plotly helper functions ─────────────────────────────────────────────────


def plot_series(*series_labels, title="", save_name=None):
    """Plot one or more pd.Series with optional save."""
    fig = go.Figure()
    for series, label, color in series_labels:
        fig.add_trace(
            go.Scatter(
                x=series.index,
                y=series.values,
                mode="lines",
                name=label,
                line=dict(color=color, width=2),
            )
        )
    apply_ppt_layout(fig, title)
    # Use the first series as reference for checkpoint date-range filtering
    _ref = series_labels[0][0] if series_labels else None
    add_checkpoints(fig, _ref)
    if save_name:
        save_fig(fig, save_name)
    fig.show()


def plot_pred(
    real,
    pred,
    title,
    real_label="Real",
    pred_label="Predictions",
    real_color=None,
    pred_color=None,
    rmse=None,
    mae=None,
    save_name=None,
):
    """Plot real vs predicted series with optional metrics annotation."""
    real_color = real_color or PPT_COLORS["train"]
    pred_color = pred_color or PPT_COLORS["pred_train"]
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=real.index,
            y=real.values,
            mode="lines",
            name=real_label,
            line=dict(color=real_color, width=2),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=pred.index,
            y=pred.values,
            mode="lines",
            name=pred_label,
            line=dict(color=pred_color, width=2, dash="dash"),
        )
    )
    if rmse is not None and mae is not None:
        fig.add_annotation(
            xref="paper",
            yref="paper",
            x=0.01,
            y=0.97,
            text=f"RMSE = {rmse:.4f}   MAE = {mae:.4f}",
            showarrow=False,
            font=dict(size=13, color=PPT_COLORS["text"]),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor=PPT_COLORS["grid"],
            borderwidth=1,
            align="left",
        )
    add_checkpoints(fig, real)
    apply_ppt_layout(fig, title)
    if save_name:
        save_fig(fig, save_name)
    fig.show()


def plot_acf_plotly(series, nlags=52, title_prefix="", save_name=None):
    """ACF plot using Plotly."""
    acf_vals, acf_ci = acf(series.dropna(), nlags=nlags, alpha=0.05)

    acf_ci_lower = acf_ci[:, 0] - acf_vals
    acf_ci_upper = acf_ci[:, 1] - acf_vals

    fig = go.Figure()

    lags = np.arange(len(acf_vals))

    # Confidence band
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([lags, lags[::-1]]),
            y=np.concatenate([acf_ci_upper, acf_ci_lower[::-1]]),
            fill="toself",
            fillcolor="rgba(33,150,243,0.15)",
            line=dict(color="rgba(0,0,0,0)"),
            showlegend=False,
        )
    )

    # Stems
    for lag, v in zip(lags, acf_vals):
        fig.add_trace(
            go.Scatter(
                x=[lag, lag],
                y=[0, v],
                mode="lines",
                line=dict(color=PPT_COLORS["train"], width=1.5),
                showlegend=False,
            )
        )

    # Dots
    fig.add_trace(
        go.Scatter(
            x=lags,
            y=acf_vals,
            mode="markers",
            marker=dict(color=PPT_COLORS["train"], size=5),
            showlegend=False,
        )
    )

    fig.add_hline(y=0, line_width=1, line_color="black")
    fig.update_layout(title=f"{title_prefix} ACF")

    apply_ppt_layout(fig, height=450)
    if save_name:
        save_fig(fig, save_name)
    fig.show()


def plot_pacf_plotly(series, nlags=52, title_prefix="", save_name=None):
    """PACF plot using Plotly."""
    pacf_vals, pacf_ci = pacf(series.dropna(), nlags=nlags, alpha=0.05)

    pacf_ci_lower = pacf_ci[:, 0] - pacf_vals
    pacf_ci_upper = pacf_ci[:, 1] - pacf_vals

    fig = go.Figure()

    lags = np.arange(len(pacf_vals))

    # Confidence band
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([lags, lags[::-1]]),
            y=np.concatenate([pacf_ci_upper, pacf_ci_lower[::-1]]),
            fill="toself",
            fillcolor="rgba(33,150,243,0.15)",
            line=dict(color="rgba(0,0,0,0)"),
            showlegend=False,
        )
    )

    # Stems
    for lag, v in zip(lags, pacf_vals):
        fig.add_trace(
            go.Scatter(
                x=[lag, lag],
                y=[0, v],
                mode="lines",
                line=dict(color=PPT_COLORS["train"], width=1.5),
                showlegend=False,
            )
        )

    # Dots
    fig.add_trace(
        go.Scatter(
            x=lags,
            y=pacf_vals,
            mode="markers",
            marker=dict(color=PPT_COLORS["train"], size=5),
            showlegend=False,
        )
    )

    fig.add_hline(y=0, line_width=1, line_color="black")
    fig.update_layout(title=f"{title_prefix} PACF")

    apply_ppt_layout(fig, height=450)
    if save_name:
        save_fig(fig, save_name)
    fig.show()


def plot_decomposition(
    original, trend, seasonality, residual, title="Decomposition", save_name=None
):
    """4-panel decomposition plot."""
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        subplot_titles=["Original", "Trend", "Seasonality", "Residual"],
        vertical_spacing=0.06,
    )

    pairs = [
        (1, original, PPT_COLORS["train"]),
        (2, trend, PPT_COLORS["trend"]),
        (3, seasonality, PPT_COLORS["season"]),
        (4, residual, PPT_COLORS["residual"]),
    ]
    for row, s, color in pairs:
        fig.add_trace(
            go.Scatter(
                x=s.index,
                y=s.values,
                mode="lines",
                line=dict(color=color, width=1.8),
                showlegend=False,
            ),
            row=row,
            col=1,
        )

    apply_ppt_layout(fig, title=title, height=900)
    fig.update_layout(margin=dict(l=60, r=40, t=100, b=60))
    if save_name:
        save_fig(fig, save_name)
    fig.show()


def plot_combined_forecast(
    train_real,
    val_real,
    test_real,
    train_pred,
    val_pred,
    test_pred,
    train_rmse,
    train_mae,
    val_rmse,
    val_mae,
    test_rmse,
    test_mae,
    title="Forecast — Train / Val / Test",
    save_name=None,
):
    """Full combined forecast chart with split lines and metric annotations."""
    fig = go.Figure()

    # Real data – continuous
    real_combined = pd.concat([train_real, val_real, test_real])
    fig.add_trace(
        go.Scatter(
            x=real_combined.index,
            y=real_combined.values,
            mode="lines",
            name="Real",
            line=dict(color=PPT_COLORS["train"], width=2),
        )
    )

    # Predictions
    for pred, label, color in [
        (train_pred, "Train pred", PPT_COLORS["pred_train"]),
        (val_pred, "Val pred", PPT_COLORS["pred_val"]),
        (test_pred, "Test pred", PPT_COLORS["pred_test"]),
    ]:
        fig.add_trace(
            go.Scatter(
                x=pred.index,
                y=pred.values,
                mode="lines",
                name=label,
                line=dict(color=color, width=2, dash="dash"),
            )
        )

    # Split lines — use add_shape + add_annotation to avoid Plotly datetime bug with add_vline
    for split_date, split_label in [
        (val_real.index[0], "Train/Val split"),
        (test_real.index[0], "Val/Test split"),
    ]:
        fig.add_shape(
            type="line",
            xref="x",
            yref="paper",
            x0=split_date,
            x1=split_date,
            y0=0,
            y1=1,
            line=dict(color=PPT_COLORS["split"], width=1.5, dash="dot"),
        )
        fig.add_annotation(
            xref="x",
            yref="paper",
            x=split_date,
            y=1.02,
            text=split_label,
            showarrow=False,
            font=dict(size=11, color=PPT_COLORS["split"]),
            xanchor="left",
        )

    # Metrics annotation
    metrics_text = (
        f"Train  RMSE={train_rmse:.4f}  MAE={train_mae:.4f}<br>"
        f"Val    RMSE={val_rmse:.4f}  MAE={val_mae:.4f}<br>"
        f"Test   RMSE={test_rmse:.4f}  MAE={test_mae:.4f}"
    )
    fig.add_annotation(
        xref="paper",
        yref="paper",
        x=0.01,
        y=0.97,
        text=metrics_text,
        showarrow=False,
        font=dict(size=12, color=PPT_COLORS["text"]),
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor=PPT_COLORS["grid"],
        borderwidth=1,
        align="left",
    )

    add_checkpoints(fig, real_combined)
    apply_ppt_layout(fig, title)
    if save_name:
        save_fig(fig, save_name)
    fig.show()

### Import data

In [52]:
# Load csv download from
# https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/precos-revenda-e-de-distribuicao-combustiveis/serie-historica-do-levantamento-de-precos
data = pd.read_excel(
    "semanal-brasil-desde-2013.xlsx",
    skiprows=17,
    usecols=["DATA INICIAL", "PREÇO MÉDIO REVENDA", "PRODUTO"],
    parse_dates=["DATA INICIAL"],
).query("PRODUTO == 'OLEO DIESEL'")

In [53]:
data = data.rename(
    columns={
        "DATA INICIAL": "Data inicial",
        "PREÇO MÉDIO REVENDA": "Preço médio revenda",
    }
)

# Sort by date
data = data.sort_values(by="Data inicial", ascending=True)
data = data.set_index("Data inicial")["Preço médio revenda"]

In [54]:
data = data.asfreq("7D")
data = data.interpolate(method="linear")

### 1. Pre-processing

In [55]:
train, tail = train_test_split(data, shuffle=False, train_size=0.5)
val, test = train_test_split(tail, shuffle=False, train_size=0.5)

In [ ]:
plot_series(
    (train, "Train", PPT_COLORS["train"]),
    (val, "Val", PPT_COLORS["val"]),
    (test, "Test", PPT_COLORS["test"]),
    title="Average resale price of diesel (R$/L)",
    save_name="01_train_val_test_split",
)

Saved: 01_train_val_test_split.html / 01_train_val_test_split.png


In [57]:
scaler = MinMaxScaler()

train_scaled = pd.Series(
    scaler.fit_transform(train.values.reshape(-1, 1)).flatten(), index=train.index
)

val_scaled = pd.Series(
    scaler.transform(val.values.reshape(-1, 1)).flatten(), index=val.index
)

test_scaled = pd.Series(
    scaler.transform(test.values.reshape(-1, 1)).flatten(), index=test.index
)

In [ ]:
plot_series(
    (train_scaled, "Train", PPT_COLORS["train"]),
    (val_scaled, "Val", PPT_COLORS["val"]),
    (test_scaled, "Test", PPT_COLORS["test"]),
    title="Average resale price of diesel (scaled)",
    save_name="02_scaled_split",
)

Saved: 02_scaled_split.html / 02_scaled_split.png


### 2. Decomposition

In [59]:
period = 52  # 52 weeks = 1 year seasonality

### 2.1 Trend

#### Moving Average

In [60]:
plot_series(
    (train_scaled, "Original", PPT_COLORS["train"]),
    (train_scaled.rolling(window=period).mean(), "Moving Average", PPT_COLORS["trend"]),
    title="Trend — Moving Average",
    save_name="03_trend_moving_average",
)

Saved: 03_trend_moving_average.html / 03_trend_moving_average.png


In [61]:
train_scaled_trend_ma = train_scaled.rolling(window=period).mean()
train_scaled_sr_ma = train_scaled - train_scaled_trend_ma

plot_series(
    (train_scaled, "Original", PPT_COLORS["train"]),
    (train_scaled_sr_ma, "Seasonality & Residual", PPT_COLORS["season"]),
    title="After removing Moving-Average Trend",
    save_name="04_sr_ma",
)

Saved: 04_sr_ma.html / 04_sr_ma.png


#### Exponential Smoothing

In [62]:
plot_series(
    (train_scaled, "Original", PPT_COLORS["train"]),
    (train_scaled.ewm(alpha=0.2).mean(), "Exponential Smoothing", PPT_COLORS["trend"]),
    title="Trend — Exponential Smoothing",
    save_name="05_trend_exp_smooth",
)

Saved: 05_trend_exp_smooth.html / 05_trend_exp_smooth.png


In [63]:
train_scaled_trend_es = train_scaled.ewm(alpha=0.2).mean()
train_scaled_sr_es = train_scaled - train_scaled_trend_es

plot_series(
    (train_scaled, "Original", PPT_COLORS["train"]),
    (train_scaled_sr_es, "Seasonality & Residual", PPT_COLORS["season"]),
    title="After removing Exponential-Smoothing Trend",
    save_name="06_sr_es",
)

Saved: 06_sr_es.html / 06_sr_es.png


### 2.2 Seasonality

In [64]:
train_scaled_smooth = train_scaled_sr_es.rolling(window=period).mean()
train_scaled_residual = train_scaled_sr_es - train_scaled_smooth

plot_series(
    (train_scaled_residual, "Residual", PPT_COLORS["residual"]),
    title="Residual (after MA smoothing of S+R)",
    save_name="07_residual_ma",
)

Saved: 07_residual_ma.html / 07_residual_ma.png


In [65]:
seasonal_pattern = [train_scaled_sr_es.iloc[i::period].mean() for i in range(period)]
train_scaled_seasonality = [
    seasonal_pattern[i % period] for i in range(len(train_scaled_sr_es))
]
train_scaled_seasonality = pd.Series(
    train_scaled_seasonality, index=train_scaled_sr_es.index
)

plot_series(
    (train_scaled_seasonality, "Seasonality (manual)", PPT_COLORS["season"]),
    title="Manual Seasonality Pattern",
    save_name="08_seasonality_manual",
)

Saved: 08_seasonality_manual.html / 08_seasonality_manual.png


In [66]:
train_scaled_residual = train_scaled_sr_es - train_scaled_seasonality

plot_series(
    (train_scaled_residual, "Residual", PPT_COLORS["residual"]),
    title="Residual (after subtracting Seasonality)",
    save_name="09_residual_final",
)

Saved: 09_residual_final.html / 09_residual_final.png


#### Residual analysis

In [67]:
plot_pacf_plotly(
    train_scaled, nlags=62, title_prefix="Train scaled", save_name="10_acf_pacf_train"
)

Saved: 10_acf_pacf_train.html / 10_acf_pacf_train.png


#### Full decomposition

In [ ]:
plot_decomposition(
    train_scaled,
    train_scaled_trend_es,
    train_scaled_seasonality,
    train_scaled_residual,
    title="Full Decomposition (Exponential Smoothing)",
    save_name="11_full_decomposition",
)

Saved: 11_full_decomposition.html / 11_full_decomposition.png


In [ ]:
train_scaled_reconstituted = (
    train_scaled_trend_es + train_scaled_seasonality + train_scaled_residual
)
plot_series(
    (train_scaled, "Original", PPT_COLORS["train"]),
    (train_scaled_reconstituted, "Reconstituted", PPT_COLORS["trend"]),
    title="Original vs Reconstituted",
    save_name="12_reconstituted",
)

Saved: 12_reconstituted.html / 12_reconstituted.png


### Automatic decomposition

In [70]:
train_decomposed = seasonal_decompose(train_scaled, period=52)

In [71]:
plot_decomposition(
    train_scaled,
    train_decomposed.trend.dropna(),
    train_decomposed.seasonal,
    train_decomposed.resid.dropna(),
    title="Automatic Decomposition (statsmodels)",
    save_name="13_auto_decomposition",
)

Saved: 13_auto_decomposition.html / 13_auto_decomposition.png


In [72]:
plot_pacf_plotly(
    train_decomposed.resid.dropna(),
    nlags=52,
    title_prefix="Decomposed residual",
    save_name="14_pacf_resid",
)

Saved: 14_pacf_resid.html / 14_pacf_resid.png


### Modeling

#### 1. ARIMA

#### Param **_d_**

In [73]:
diff_number = find_differencing_order(train_scaled, 0.05)
train_scaled_diff = train_scaled.diff(diff_number).dropna()

d=0, p-value=0.84738
d=1, p-value=0.00000


#### Param **_p_**

In [74]:
plot_pacf_plotly(
    train_scaled_diff, nlags=62, title_prefix="PACF (p param)", save_name="15_pacf_p"
)

Saved: 15_pacf_p.html / 15_pacf_p.png


#### Param **_q_**

In [75]:
plot_acf_plotly(
    train_scaled_diff, nlags=62, title_prefix="ACF (q param)", save_name="16_acf_q"
)

Saved: 16_acf_q.html / 16_acf_q.png


In [76]:
from statsmodels.stats.diagnostic import acorr_ljungbox

for order in [(1, 1, 1), (13, 1, 1), (1, 1, 13)]:
    m = ARIMA(train_scaled, order=order, freq="7D").fit()
    lb = acorr_ljungbox(m.resid.dropna(), lags=[13], return_df=True)
    print(
        f"ARIMA{order} - AIC={m.aic:.2f} - BIC={m.bic:.2f} - LB p={lb['lb_pvalue'].values[0]:.4f}"
    )

ARIMA(1, 1, 1) - AIC=-1771.43 - BIC=-1759.92 - LB p=0.6127
ARIMA(13, 1, 1) - AIC=-1762.11 - BIC=-1704.54 - LB p=0.9985
ARIMA(1, 1, 13) - AIC=-1755.16 - BIC=-1697.60 - LB p=0.9975


In [77]:
p = 1
q = 1
d = 1

In [78]:
model = ARIMA(train_scaled, order=(p, d, q), freq="7D").fit()

In [79]:
train_pred = model.predict(train_scaled.index[0], train_scaled.index[-1])
train_rmse = root_mean_squared_error(train_scaled, train_pred)
train_mae = mean_absolute_error(train_scaled, train_pred)

plot_pred(
    train_scaled,
    train_pred,
    title="ARIMA — Training data",
    rmse=train_rmse,
    mae=train_mae,
    save_name="17_arima_train",
)

Saved: 17_arima_train.html / 17_arima_train.png


In [80]:
val_pred_arima = model.predict(val_scaled.index[0], val_scaled.index[-1])
val_rmse = root_mean_squared_error(val_scaled, val_pred_arima)
val_mae = mean_absolute_error(val_scaled, val_pred_arima)

plot_pred(
    val_scaled,
    val_pred_arima,
    title="ARIMA — Default forecast (val)",
    real_color=PPT_COLORS["val"],
    pred_color=PPT_COLORS["pred_val"],
    rmse=val_rmse,
    mae=val_mae,
    save_name="18_arima_val_default",
)

Saved: 18_arima_val_default.html / 18_arima_val_default.png


In [81]:
forecasts_val = forecast(train_scaled, val_scaled, (1, 1, 1))
val_rmse_f = root_mean_squared_error(val_scaled, forecasts_val)
val_mae_f = mean_absolute_error(val_scaled, forecasts_val)

plot_pred(
    val_scaled,
    forecasts_val,
    title="ARIMA — Online forecast (val)",
    real_color=PPT_COLORS["val"],
    pred_color=PPT_COLORS["pred_val"],
    rmse=val_rmse_f,
    mae=val_mae_f,
    save_name="19_arima_val_online",
)

# ── ARIMA Test predictions ────────────────────────────────────────────────
forecasts_test = forecast(pd.concat([train_scaled, val_scaled]), test_scaled, (1, 1, 1))
test_rmse_arima = root_mean_squared_error(test_scaled, forecasts_test)
test_mae_arima = mean_absolute_error(test_scaled, forecasts_test)

plot_pred(
    test_scaled,
    forecasts_test,
    title="ARIMA — Online forecast (test)",
    real_color=PPT_COLORS["test"],
    pred_color=PPT_COLORS["pred_test"],
    rmse=test_rmse_arima,
    mae=test_mae_arima,
    save_name="20_arima_test",
)

Saved: 19_arima_val_online.html / 19_arima_val_online.png


Saved: 20_arima_test.html / 20_arima_test.png


In [82]:
# Retrain train_pred & forecasts_val for combined plot (reuse from above)
train_pred = model.predict(train_scaled.index[0], train_scaled.index[-1])
train_rmse = root_mean_squared_error(train_scaled, train_pred)
train_mae = mean_absolute_error(train_scaled, train_pred)
forecasts_val = forecast(train_scaled, val_scaled, (1, 1, 1))
val_rmse_f = root_mean_squared_error(val_scaled, forecasts_val)
val_mae_f = mean_absolute_error(val_scaled, forecasts_val)

plot_combined_forecast(
    train_scaled,
    val_scaled,
    test_scaled,
    train_pred,
    forecasts_val,
    forecasts_test,
    train_rmse,
    train_mae,
    val_rmse_f,
    val_mae_f,
    test_rmse_arima,
    test_mae_arima,
    title="ARIMA — Training, Validation & Test",
    save_name="21_arima_combined",
)

Saved: 21_arima_combined.html / 21_arima_combined.png


### Non-linear models

In [83]:
significant_lags = significant_lags_acf(train_scaled.diff().dropna(), max_lag=52)
significant_lags

[1, 13, 14, 26, 45]

#### SVR

In [84]:
window_size = 45

(X_train, y_train), (X_val, y_val) = window_data(window_size, train_scaled, val_scaled)
y_train = pd.Series(y_train, index=train_scaled.iloc[window_size:].index)
y_val = pd.Series(y_val, index=val_scaled.index)

# ──  Window test data (needs train+val as history) ───────────────────────
(_, _), (X_test_svr, y_test_svr_arr) = window_data(
    window_size,
    pd.concat([train_scaled, val_scaled]),
    test_scaled,
)
y_test_svr = pd.Series(y_test_svr_arr, index=test_scaled.index)

In [85]:
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.01, 0.1, 0.5],
    "kernel": ["rbf", "linear"],
}

grid = GridSearchCV(SVR(), param_grid, cv=tscv, scoring="neg_root_mean_squared_error")
grid.fit(X_train, y_train)

svr = grid.best_estimator_
print(grid.best_params_)

{'C': 10, 'epsilon': 0.01, 'kernel': 'linear'}


In [86]:
pred_train_svr = pd.Series(data=svr.predict(X_train), index=y_train.index)
pred_val_svr = pd.Series(svr.predict(X_val), index=y_val.index)
pred_test_svr = pd.Series(svr.predict(X_test_svr), index=y_test_svr.index)

In [87]:
train_svr_rmse = root_mean_squared_error(y_train, pred_train_svr)
train_svr_mae = mean_absolute_error(y_train, pred_train_svr)

plot_pred(
    train_scaled,
    pred_train_svr,
    title="SVR — Training data",
    rmse=train_svr_rmse,
    mae=train_svr_mae,
    save_name="22_svr_train",
)

Saved: 22_svr_train.html / 22_svr_train.png


In [88]:
val_svr_rmse = root_mean_squared_error(y_val, pred_val_svr)
val_svr_mae = mean_absolute_error(y_val, pred_val_svr)

plot_pred(
    val_scaled,
    pred_val_svr,
    title="SVR — Validation data",
    real_color=PPT_COLORS["val"],
    pred_color=PPT_COLORS["pred_val"],
    rmse=val_svr_rmse,
    mae=val_svr_mae,
    save_name="23_svr_val",
)

# ── SVR Test predictions ──────────────────────────────────────────────────
test_svr_rmse = root_mean_squared_error(y_test_svr, pred_test_svr)
test_svr_mae = mean_absolute_error(y_test_svr, pred_test_svr)

plot_pred(
    test_scaled,
    pred_test_svr,
    title="SVR — Test data",
    real_color=PPT_COLORS["test"],
    pred_color=PPT_COLORS["pred_test"],
    rmse=test_svr_rmse,
    mae=test_svr_mae,
    save_name="24_svr_test",
)

plot_combined_forecast(
    train_scaled,
    val_scaled,
    test_scaled,
    pred_train_svr,
    pred_val_svr,
    pred_test_svr,
    train_svr_rmse,
    train_svr_mae,
    val_svr_rmse,
    val_svr_mae,
    test_svr_rmse,
    test_svr_mae,
    title="SVR — Training, Validation & Test",
    save_name="25_svr_combined",
)

Saved: 23_svr_val.html / 23_svr_val.png


Saved: 24_svr_test.html / 24_svr_test.png


Saved: 25_svr_combined.html / 25_svr_combined.png


#### LSTM

In [89]:
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"Using device: {DEVICE} | AMP: {USE_AMP}")


class LSTMModel(nn.Module):
    def __init__(
        self, n_features=1, hidden_size=64, activation="relu", num_layers=2, dropout=0.2
    ):
        super().__init__()
        # If num_layers is 1, PyTorch's LSTM dropout doesn't work. Set to 0 to stop warnings.
        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            n_features,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )

        # Deeper head for regression
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LeakyReLU() if activation == "relu" else nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),  # Linear output for regression
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.regressor(out[:, -1, :])


def make_loader(X_t, y_t, indices, batch_size: int = 64) -> DataLoader:
    """Build a single DataLoader from a tensor index slice."""
    kwargs = dict(
        batch_size=batch_size,
        shuffle=False,
        pin_memory=DEVICE.type == "cuda",
        num_workers=0,
    )
    return DataLoader(TensorDataset(X_t[indices], y_t[indices]), **kwargs)


def make_tensors(X, y) -> tuple[torch.Tensor, torch.Tensor]:
    """Convert raw arrays to float tensors once."""
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_t = torch.tensor(y.values, dtype=torch.float32).unsqueeze(-1)
    return X_t, y_t


def train_model(
    train_loader: DataLoader,
    params: dict,
    device: torch.device = DEVICE,
) -> nn.Module:
    model = LSTMModel(
        hidden_size=params["hidden_size"],
        activation=params["activation"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer_cls = (
        torch.optim.Adam if params["optimizer"] == "adam" else torch.optim.SGD
    )
    opt = optimizer_cls(model.parameters(), lr=params["lr"])
    scaler = GradScaler() if USE_AMP else None

    for _ in range(params["epochs"]):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            if USE_AMP:
                with autocast(device_type="cuda"):
                    loss = criterion(model(X_batch), y_batch)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                criterion(model(X_batch), y_batch).backward()
                opt.step()

    return model


def evaluate(
    model: nn.Module,
    val_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device = DEVICE,
) -> float:
    model.eval()
    total_loss = 0.0
    with torch.no_grad(), autocast(device_type=device.type, enabled=USE_AMP):
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            total_loss += criterion(model(X_batch), y_batch).item()
    return total_loss / len(val_loader)


def objective(
    trial: optuna.Trial,
    X_t: torch.Tensor,
    y_t: torch.Tensor,
    tscv: TimeSeriesSplit,
    device: torch.device = DEVICE,
) -> float:
    params = {
        "hidden_size": trial.suggest_categorical("hidden_size", [64, 96]),
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "dropout": trial.suggest_float("dropout", 0.0, 0.2),
        "epochs": trial.suggest_categorical("epochs", [100, 200]),
        "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "sgd"]),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
    }

    criterion = nn.MSELoss()
    fold_losses = []

    for train_idx, val_idx in tscv.split(X_t):
        train_loader = make_loader(X_t, y_t, train_idx)
        val_loader = make_loader(X_t, y_t, val_idx)
        model = train_model(train_loader, params, device)
        fold_losses.append(evaluate(model, val_loader, criterion, device))

    return float(torch.tensor(fold_losses).mean())


def fit_lstm(
    X,
    y,
    n_trials: int = 5,
    n_splits: int = 2,
    device: torch.device = DEVICE,
) -> nn.Module:
    X_t, y_t = make_tensors(X, y)
    tscv = TimeSeriesSplit(n_splits=n_splits)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )
    study.optimize(
        lambda trial: objective(trial, X_t, y_t, tscv, device),
        n_trials=n_trials,
        n_jobs=1,
    )

    best_params = study.best_params
    print(f"Best params: {best_params}")

    full_loader = make_loader(X_t, y_t, indices=range(len(X_t)))
    return train_model(full_loader, best_params, device)


def lstm_predict(
    model: nn.Module,
    X,
    index,
    device: torch.device = DEVICE,
) -> pd.Series:
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1).to(device)
        preds = model(X_t).squeeze().cpu().numpy()
    return pd.Series(preds, index=index)


lags = [45, 26, 14, 13, 1]

# ── LSTM per window-size: train, val AND test ──────────────────────────────
for window_size in lags:
    # Train / Val windows
    (X_train_l, y_train_arr), (X_val_l, y_val_arr) = window_data(
        window_size, train_scaled, val_scaled
    )
    y_train_l = pd.Series(y_train_arr, index=train_scaled.iloc[window_size:].index)
    y_val_l = pd.Series(y_val_arr, index=val_scaled.index)

    # Test window (history = train + val)
    (_, _), (X_test_l, y_test_arr) = window_data(
        window_size,
        pd.concat([train_scaled, val_scaled]),
        test_scaled,
    )
    y_test_l = pd.Series(y_test_arr, index=test_scaled.index)

    # Fit on train, predict val & test
    model_lstm = fit_lstm(X_train_l, y_train_l, n_trials=15)

    pred_train_l = lstm_predict(model_lstm, X_train_l, y_train_l.index)
    pred_val_l = lstm_predict(model_lstm, X_val_l, y_val_l.index)
    pred_test_l = lstm_predict(model_lstm, X_test_l, y_test_l.index)

    train_rmse_l = root_mean_squared_error(y_train_l, pred_train_l)
    train_mae_l = mean_absolute_error(y_train_l, pred_train_l)
    val_rmse_l = root_mean_squared_error(y_val_l, pred_val_l)
    val_mae_l = mean_absolute_error(y_val_l, pred_val_l)
    test_rmse_l = root_mean_squared_error(y_test_l, pred_test_l)
    test_mae_l = mean_absolute_error(y_test_l, pred_test_l)

    # Val plot
    plot_pred(
        y_val_l,
        pred_val_l,
        title=f"LSTM  window={window_size} — Validation",
        real_color=PPT_COLORS["val"],
        pred_color=PPT_COLORS["pred_val"],
        rmse=val_rmse_l,
        mae=val_mae_l,
        save_name=f"26_lstm_val_w{window_size}",
    )

    # Test plot
    plot_pred(
        y_test_l,
        pred_test_l,
        title=f"LSTM  window={window_size} — Test",
        real_color=PPT_COLORS["test"],
        pred_color=PPT_COLORS["pred_test"],
        rmse=test_rmse_l,
        mae=test_mae_l,
        save_name=f"27_lstm_test_w{window_size}",
    )

    # Combined train/val/test
    plot_combined_forecast(
        train_scaled,
        val_scaled,
        test_scaled,
        pred_train_l,
        pred_val_l,
        pred_test_l,
        train_rmse_l,
        train_mae_l,
        val_rmse_l,
        val_mae_l,
        test_rmse_l,
        test_mae_l,
        title=f"LSTM  window={window_size} — Train / Val / Test",
        save_name=f"28_lstm_combined_w{window_size}",
    )

Using device: cuda | AMP: True
Best params: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.009045457782107613, 'epochs': 200, 'activation': 'tanh', 'optimizer': 'adam', 'lr': 0.001217284708112243}
Saved: 26_lstm_val_w45.html / 26_lstm_val_w45.png


Saved: 27_lstm_test_w45.html / 27_lstm_test_w45.png


Saved: 28_lstm_combined_w45.html / 28_lstm_combined_w45.png


Best params: {'hidden_size': 96, 'num_layers': 1, 'dropout': 0.10401360423556216, 'epochs': 100, 'activation': 'relu', 'optimizer': 'adam', 'lr': 0.0015696396388661157}
Saved: 26_lstm_val_w26.html / 26_lstm_val_w26.png


Saved: 27_lstm_test_w26.html / 27_lstm_test_w26.png


...

Saved: 27_lstm_test_w13.html / 27_lstm_test_w13.png


Saved: 28_lstm_combined_w13.html / 28_lstm_combined_w13.png


Best params: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.009045457782107613, 'epochs': 200, 'activation': 'tanh', 'optimizer': 'adam', 'lr': 0.001217284708112243}
Saved: 26_lstm_val_w1.html / 26_lstm_val_w1.png


Saved: 27_lstm_test_w1.html / 27_lstm_test_w1.png


Saved: 28_lstm_combined_w1.html / 28_lstm_combined_w1.png
